#Generation d'une simulation de combat

In [ ]:
import random

# matplotlib est requis pour les graphiques (déjà préinstallé sur Google Colab)
!pip install -q matplotlib
import matplotlib.pyplot as plt

# NOTE : faker n'est plus utilisé (les prénoms viennent de STATS_REGION),
# l'installation et l'import ont été retirés pour accélérer le démarrage.

## Création des humains

In [ ]:
import random
import itertools

class Human:
    _counter = 0
    # LOI DU REEL (demographie) : 10 regions et leur part de la population mondiale en
    # 2025 ('part', en %). Asie = Est + Sud + Sud-Est + Moyen-Orient = 58,7 % ; Afrique =
    # Nord + subsaharienne = 18,8 % (chiffres ONU 2025). Avant : 6 regions, 51 % pour
    # l'Asie, 23,5 % pour l'Afrique, ni Asie du Sud-Est ni Moyen-Orient.
    # 'Asie_Sud' inclut l'Asie centrale ; 'Moyen_Orient' = Asie de l'Ouest.
    # taille_h / taille_f : taille adulte moyenne (cm). imc_median : IMC median adulte.
    STATS_REGION = {
        'Europe': {'part': 9.0, 'taille_h': 178, 'taille_f': 165, 'imc_median': 26.0, 'eco_factor': 40000, 'prenoms_h': ['Jean', 'Pierre', 'Luc', 'Hans', 'Marco', 'Ivan'], 'prenoms_f': ['Marie', 'Sophie', 'Claire', 'Anna', 'Giulia', 'Olga']},
        'Amerique_Nord': {'part': 4.7, 'taille_h': 176, 'taille_f': 163, 'imc_median': 28.0, 'eco_factor': 60000, 'prenoms_h': ['John', 'Michael', 'James', 'David'], 'prenoms_f': ['Emma', 'Sarah', 'Jessica', 'Ashley']},
        'Asie_Est': {'part': 20.1, 'taille_h': 172, 'taille_f': 160, 'imc_median': 23.5, 'eco_factor': 25000, 'prenoms_h': ['Takeshi', 'Kenji', 'Wei', 'Jun', 'Min-jun'], 'prenoms_f': ['Sakura', 'Yui', 'Mei', 'Li', 'Ji-woo']},
        'Asie_Sud': {'part': 26.2, 'taille_h': 166, 'taille_f': 153, 'imc_median': 22.0, 'eco_factor': 4000, 'prenoms_h': ['Raj', 'Amit', 'Arjun', 'Imran', 'Rahim'], 'prenoms_f': ['Priya', 'Ananya', 'Deepika', 'Ayesha', 'Fatima']},
        'Asie_Sud_Est': {'part': 8.6, 'taille_h': 165, 'taille_f': 154, 'imc_median': 23.0, 'eco_factor': 6000, 'prenoms_h': ['Budi', 'Nguyen', 'Somchai', 'Jose'], 'prenoms_f': ['Siti', 'Linh', 'Maria', 'Dewi']},
        'Moyen_Orient': {'part': 3.8, 'taille_h': 173, 'taille_f': 160, 'imc_median': 27.0, 'eco_factor': 15000, 'prenoms_h': ['Ali', 'Omar', 'Mehmet', 'Yusuf'], 'prenoms_f': ['Leila', 'Zeynep', 'Noor', 'Maryam']},
        'Afrique_Nord': {'part': 3.3, 'taille_h': 172, 'taille_f': 159, 'imc_median': 27.0, 'eco_factor': 5000, 'prenoms_h': ['Mohammed', 'Ahmed', 'Karim'], 'prenoms_f': ['Amina', 'Salma', 'Yasmine']},
        'Afrique_Subsaharienne': {'part': 15.5, 'taille_h': 168, 'taille_f': 158, 'imc_median': 23.0, 'eco_factor': 1500, 'prenoms_h': ['Amadou', 'Kofi', 'Chinedu', 'Tendai'], 'prenoms_f': ['Fatou', 'Ngozi', 'Amara', 'Wanjiru']},
        'Amerique_Latine': {'part': 8.1, 'taille_h': 171, 'taille_f': 158, 'imc_median': 27.0, 'eco_factor': 8000, 'prenoms_h': ['Carlos', 'Jose', 'Luis', 'Joao'], 'prenoms_f': ['Maria', 'Sofia', 'Isabella', 'Camila']},
        'Oceanie': {'part': 0.6, 'taille_h': 176, 'taille_f': 163, 'imc_median': 27.5, 'eco_factor': 45000, 'prenoms_h': ['Jack', 'Liam', 'Oliver'], 'prenoms_f': ['Charlotte', 'Olivia', 'Mia']}
    }
    CLASSES_SOCIALES = ['Indigent', 'Misérable', 'Pauvre', 'Modeste', 'Classe moyenne inf.', 'Classe moyenne sup.', 'Aisé', 'Riche', 'Très riche', 'Fortune']

    # OPTIMISATION : la liste des clés de région et les poids sont calculés UNE FOIS
    # au niveau de la classe, au lieu d'être reconstruits à chaque Human().
    REGION_KEYS = list(STATS_REGION.keys())
    # Les poids sont LUS dans STATS_REGION ('part') : plus de liste séparée dont les
    # commentaires pouvaient contredire les valeurs.
    REGION_WEIGHTS = [region['part'] for region in STATS_REGION.values()]
    # OPTIMISATION : poids cumulés précalculés -> random.choices(cum_weights=...) évite
    # de re-sommer la distribution à chaque tirage (8192 appels à la génération).
    REGION_CUM_WEIGHTS = list(itertools.accumulate(REGION_WEIGHTS))

    # OPTIMISATION : tranches d'âge et poids cumulés hissés en constantes de classe.
    # Avant, deux listes littérales étaient ré-allouées à chaque _calculer_age().
    # BUGFIX (pyramide a trous) : les 4 tranches (enfant/adulte/mature/vieux) et leurs
    # lois triangulaires creusaient des trous (61-65 ans : 1,0 % ; 66-70 ans : 2,7 %)
    # et mettaient plus de 41-45 ans que de 18-22 ans. On tire desormais l'age dans la
    # pyramide mondiale par tranches de 5 ans (ordre de grandeur ONU, World Population
    # Prospects 2024 : 0-14 ans ~25 %, 65 ans et + ~10 %), uniforme dans la tranche.
    AGE_TRANCHES = list(range(0, 105, 5))   # debut de chaque tranche : 0-4, 5-9, ... 100-104
    AGE_POIDS = [8.1, 8.4, 8.3, 8.0, 7.6, 7.4, 7.5, 7.2, 6.5, 5.9, 5.6, 5.2, 4.4,
                 3.6, 2.8, 1.8, 1.1, 0.6, 0.25, 0.07, 0.01]
    AGE_CUM_WEIGHTS = list(itertools.accumulate(AGE_POIDS))

    # LOI DU REEL (richesse) : multiplicateur de la probabilite d'etre entraine, par
    # classe sociale (de Indigent a Fortune).
    ACCES_ENTRAINEMENT = [0.3, 0.5, 0.7, 0.9, 1.0, 1.2, 1.5, 1.8, 2.0, 2.0]
    # Classes sociales qui ont acces aux soins entre deux rounds (Classe moyenne sup.
    # et au-dessus) : equivalent d'une equipe medicale personnelle.
    INDEX_ACCES_SOINS = 5

    # Frequence du handicap par age (age maximal de la tranche, probabilite).
    # Moyenne ponderee par la pyramide des ages : ~16 %.
    HANDICAP_PAR_AGE = [(17, 0.07), (39, 0.10), (59, 0.18), (74, 0.38), (84, 0.55), (200, 0.75)]

    # Part de la taille adulte atteinte a chaque age (points de la courbe de croissance).
    CROISSANCE = [(0, 0.29), (2, 0.49), (5, 0.62), (10, 0.78), (14, 0.93), (18, 1.0)]

    # OPTIMISATION (#5) : proportions de PV par membre (constante de classe, partagee
    # par _construire_corps -> _initialiser_corps et integrer_joueur).
    PROPORTIONS_CORPS = {
        'tete': 0.8, 'torse': 1.0, 'bras_g': 0.4, 'bras_d': 0.4, 'jambe_g': 0.5, 'jambe_d': 0.5
    }

    def __init__(self):
        Human._counter += 1
        self.id = Human._counter
        self.region_key = random.choices(Human.REGION_KEYS, cum_weights=Human.REGION_CUM_WEIGHTS, k=1)[0]
        region_data = self.STATS_REGION[self.region_key]

        # OPTIMISATION : random.random() au lieu de np.random.choice (beaucoup plus
        # rapide pour un tirage binaire scalaire), distribution identique.
        self.sexe = 'Homme' if random.random() < 0.505 else 'Femme'
        self._generer_identite(region_data)
        self._calculer_age()
        self._calculer_physique(region_data)
        self._calculer_sante()

        # --- AJOUT ICI ---
        self._calculer_mental()

        self._calculer_stats_combat()
        # LOI DU REEL (richesse) : la richesse est tiree AVANT l'entrainement, car elle
        # ouvre l'acces a un club ou a un coach et aux soins (cf. acces_soins).
        self._calculer_richesse(region_data)
        self._calculer_entrainement()
        self._initialiser_corps()

        # BUGFIX : l'incapacité liée au grand âge est tirée UNE SEULE FOIS à la création.
        # Avant, elle était re-tirée à chaque appel de recuperation(), ce qui rendait
        # la capacité de combat aléatoire d'un round à l'autre.
        # LOI DU REEL (sante) : elle passe désormais par le handicap (_calculer_sante),
        # tiré une seule fois lui aussi.
        self._actualiser_stats()
        self._verifier_capacite_combat()

    def _generer_identite(self, region_data):
        if self.sexe == 'Homme': self.prenom = random.choice(region_data['prenoms_h'])
        else: self.prenom = random.choice(region_data['prenoms_f'])

    def _calculer_age(self):
        debut = random.choices(Human.AGE_TRANCHES, cum_weights=Human.AGE_CUM_WEIGHTS, k=1)[0]
        self.age = debut + random.randrange(5)

    def _calculer_physique(self, region_data):
        base_taille = region_data['taille_h'] if self.sexe == 'Homme' else region_data['taille_f']
        # BUGFIX : suppression d'une ligne morte (facteur_croissance était calculé puis
        # systématiquement écrasé pour age<18, et valait toujours 1.0 pour age>=18).
        # LOI DU REEL (corpulence) : courbe de croissance par points (un nouveau-né mesure
        # 29 % de sa taille adulte, pas 50 %) et écart-type de taille réaliste : 4 % de
        # la taille, soit ~7 cm chez l'adulte (3 cm avant).
        facteur_croissance = 1.0
        if self.age < 18:
            for (a0, f0), (a1, f1) in zip(Human.CROISSANCE, Human.CROISSANCE[1:]):
                if self.age <= a1:
                    facteur_croissance = Human._interp(self.age, a0, f0, a1, f1)
                    break
        taille_moyenne = base_taille * facteur_croissance
        self.taille = max(40.0, random.gauss(taille_moyenne, 0.04 * taille_moyenne))

        # LOI DU REEL (corpulence) : l'IMC suit une loi log-normale (queue vers l'obésité)
        # autour d'une médiane régionale. Avant, un gauss(imc, 2) donnait 1,8 %
        # d'adultes obèses et aucun maigre ; l'OMS compte ~16 % d'adultes obèses (2022)
        # et ~8 % d'adultes en insuffisance pondérale.
        if self.age < 18:
            imc_median = 16.5 if self.age < 2 else 15.5 + 0.35 * max(0, self.age - 5)
            sigma = 0.12
        else:
            # La corpulence monte de 18 à 50 ans, puis redescend au grand âge.
            imc_median = region_data['imc_median'] + Human._interp(self.age, 18, -1.2, 50, 1.6)
            if self.age > 70: imc_median -= Human._interp(self.age, 70, 0.0, 90, 1.5)
            # Les femmes sont plus dispersées (plus d'obésité ET plus de maigreur).
            sigma = 0.165 if self.sexe == 'Homme' else 0.20
        imc = max(11.0, min(65.0, imc_median * random.lognormvariate(0, sigma)))
        self.poids = round(imc * (self.taille/100)**2, 1)

    def _calculer_sante(self):
        # 3.4 : courbe acceleree, prevalence realiste (jeunes ~2%, tres ages ~50%).
        # LOI DU REEL (sante) : on separe deux maladies. AIGUE (infection, grippe...) :
        # frappe a tout age et guerit avec le repos. CHRONIQUE (coeur, diabete,
        # poumons, cancer...) : sa frequence monte avec l'age et elle ne guerit PAS en
        # quelques jours. Avant, la maladie liee a l'age guerissait toute seule : sans
        # medecin, 100 % des malades (age moyen 53 ans) etaient gueris en 8 jours.
        self.maladie_chronique = random.random() < (self.age / 100) ** 2 * 0.5
        self.est_malade = self.maladie_chronique or random.random() < 0.04
        self.severite_maladie = min(1.0, random.expovariate(1 / 0.3)) if self.est_malade else 0.0
        # Un traitement soulage une maladie chronique sans la guerir : plancher a 60 %
        # de la severite initiale (atteint seulement avec une equipe medicale).
        self._severite_plancher = self.severite_maladie * 0.6 if self.maladie_chronique else 0.0

        # LOI DU REEL (sante) : handicap durable (moteur, sensoriel, cognitif...).
        # L'OMS compte 1,3 milliard de personnes avec un handicap significatif, soit
        # 16 % de l'humanite, surtout aux grands ages. Avant : aucun, hors une regle
        # « plus de 85 ans = 80 % d'incapables ». Severite de 0 a 1 ; au-dela de 0.8
        # la personne ne peut pas se battre.
        self.handicap = 0.0
        for age_max, proba in Human.HANDICAP_PAR_AGE:
            if self.age <= age_max:
                if random.random() < proba:
                    self.handicap = random.betavariate(3, 2) if self.age >= 75 else random.betavariate(2, 3)
                break

    # --- NOUVELLE METHODE ---
    def _calculer_mental(self):
        base = random.gauss(0.85, 0.15) # La plupart sont stables
        if self.age < 12: base -= 0.2
        if self.age > 80: base -= 0.1
        if self.est_malade: base -= self.severite_maladie * 0.3
        self.mental = max(0.1, min(1.0, base))
        self.mental_max = self.mental
        # 5.1 : resilience psychologique individuelle (encaisse plus ou moins le
        # trauma). Variabilite via betavariate, attenuee aux ages extremes.
        res = random.betavariate(2, 2)
        if self.age < 14: res *= 0.7
        if self.age > 75: res *= 0.85
        self.resilience = max(0.05, min(1.0, res))

    def _facteur_maladie(self):
        # BUGFIX #3 : malus multiplicatif REVERSIBLE lie a la maladie. Vaut 1.0 une
        # fois gueri, ce qui permet a recuperation() de restaurer la pleine stat.
        return (1 - self.severite_maladie * 0.5) if self.est_malade else 1.0

    @staticmethod
    def _interp(x, x0, y0, x1, y1):
        # Interpolation lineaire bornee : y0 avant x0, y1 apres x1.
        if x <= x0: return y0
        if x >= x1: return y1
        return y0 + (y1 - y0) * (x - x0) / (x1 - x0)

    @staticmethod
    def taux_gras(age, sexe, imc):
        """
        Part de masse grasse (0-1) estimee par l'IMC, l'age et le sexe : formules de
        Deurenberg, Weststrate & Seidell (1991), enfants (<= 15 ans) et adultes.
        """
        homme = 1 if sexe == 'Homme' else 0
        if age < 16: pourcent = 1.51 * imc - 0.70 * age - 3.6 * homme + 1.4
        else: pourcent = 1.20 * imc + 0.23 * age - 10.8 * homme - 5.4
        return max(0.05, min(0.60, pourcent / 100))

    @staticmethod
    def stats_attendues(age, sexe, taille, poids):
        """
        Force et vitesse MOYENNES d'un gabarit donne (aucun tirage aleatoire).

        LOI DU REEL : l'ancien socle gauss(30, 10) ne dependait ni du sexe ni de l'age.
        Resultat : une femme a 92 % de la force d'un homme, un enfant de 9 ans a 73 %,
        un octogenaire a 99 %. La force part maintenant de la MASSE MAIGRE :
          force = masse maigre x coefficient de sexe x maturite x declin de l'age
        """
        interp = Human._interp
        imc = poids / (taille / 100) ** 2
        masse_maigre = poids * (1 - Human.taux_gras(age, sexe, imc))

        # SEXE : a masse maigre egale, une femme adulte developpe moins de force (masse
        # musculaire davantage dans le bas du corps). Miller et al. 1993 : 52 % de la
        # force des hommes pour le haut du corps, 66 % pour le bas. Avec l'ecart de
        # masse maigre (~0,73), 0.82 donne ~0,60 au total. Ecart nul avant la puberte.
        if sexe == 'Homme': coef_sexe_force, coef_sexe_vitesse = 1.0, 1.0
        else: coef_sexe_force, coef_sexe_vitesse = interp(age, 11, 0.97, 17, 0.82), interp(age, 11, 1.0, 17, 0.93)

        # ENFANTS : a masse maigre egale un enfant est moins fort (maturation
        # neuromusculaire). Poigne a 9 ans : ~1/3 de celle d'un homme adulte.
        maturite = interp(age, 0, 0.5, 18, 1.0)
        # VIEILLESSE : pic entre 30 et 39 ans (normes mondiales de poigne), puis declin
        # qui s'accelere. Avec la fonte de masse maigre (taux_gras monte avec l'age), un
        # homme de 80-90 ans garde ~55-60 % de la force d'un jeune adulte (etudes de
        # poigne sur 20-91 ans : 52 a 71 %).
        if age <= 40: declin = interp(age, 18, 0.96, 35, 1.03)
        elif age <= 60: declin = interp(age, 40, 1.03, 60, 0.92)
        elif age <= 80: declin = interp(age, 60, 0.92, 80, 0.72)
        else: declin = max(0.40, 0.72 - (age - 80) * 0.012)
        force = max(1.0, masse_maigre * coef_sexe_force * maturite * declin)

        # Vitesse : un enfant de 9 ans court a ~70 % de la vitesse d'un adulte ; pic de
        # 18 a 30 ans ; -0,6 %/an jusqu'a 60 ans, -1,2 %/an ensuite.
        if age < 18: facteur_age_vitesse = interp(age, 0, 0.45, 18, 1.0)
        elif age <= 30: facteur_age_vitesse = 1.0
        elif age <= 60: facteur_age_vitesse = interp(age, 30, 1.0, 60, 0.82)
        else: facteur_age_vitesse = max(0.35, 0.82 - (age - 60) * 0.012)
        # BUGFIX ("lourd = lent") : le surpoids coute de la vitesse (-2,5 % par point
        # d'IMC au-dela de 25, plancher 50 %). Avant, 60 kg et 150 kg couraient pareil.
        facteur_poids_vitesse = 1.0 if imc <= 25 else max(0.5, 1 - (imc - 25) * 0.025)
        vitesse = 50 * facteur_age_vitesse * facteur_poids_vitesse * coef_sexe_vitesse
        return force, vitesse

    @staticmethod
    def robustesse(age):
        """
        Solidite du corps (multiplie les PV). LOI DU REEL : un enfant et un vieillard
        encaissent moins qu'un adulte de meme poids (squelette immature, puis os
        fragiles apres 50 ans).
        """
        if age < 18: return Human._interp(age, 0, 0.6, 18, 1.0)
        return Human._interp(age, 50, 1.0, 90, 0.70)

    # Gain maximal apporte par l'entrainement (niveau 1.0). Multiplicatif : un
    # entrainement developpe la force qu'on a, il n'ajoute pas 20 points a tout le
    # monde (l'ancien +20 valait +25 % pour un homme mais +45 % pour une femme legere).
    GAIN_FORCE_ENTRAINEMENT = 0.35
    GAIN_VITESSE_ENTRAINEMENT = 0.20

    def _calculer_stats_combat(self):
        # BUGFIX : force bornee a >= 1. Une force negative (tirage gaussien bas +
        # faible poids) produisait des degats negatifs qui SOIGNAIENT la cible.
        # OPTIMISATION (#1) : random.gauss au lieu de np.random.normal (tirage scalaire
        # ~10x plus rapide, distribution identique) sur toute la generation.
        # 3.3 : seule la masse "utile" (maigre) fait la force. Au-dela d'un IMC
        # athletique (~25), le poids en plus est surtout du gras et ne compte qu'a
        # 20% -> un obese n'a pas une force de colosse.
        # BUGFIX (joueur tire aux des) : la partie DETERMINISTE du modele vit dans
        # stats_attendues() ; la population y ajoute son alea individuel, le joueur
        # (integrer_joueur) prend la valeur attendue telle quelle.
        # L'alea individuel est multiplicatif (log-normal, ~15 %) : il ne peut plus
        # donner a un enfant la force d'un adulte, ni une force negative.
        force_moy, vitesse_moy = Human.stats_attendues(self.age, self.sexe, self.taille, self.poids)
        self.force = max(1.0, force_moy * random.lognormvariate(0, 0.15))
        self.vitesse = vitesse_moy * random.lognormvariate(0, 0.15)
        # LOI DU REEL (entrainement) : le QI est norme PAR AGE (moyenne 100 a tout age).
        # Avant, +0,2 point par an apres 20 ans donnait 113 a 80 ans et plus. Ici,
        # `intelligence` sert au sang-froid tactique : il murit pendant l'enfance et
        # baisse au grand age (declin cognitif, demence).
        self.intelligence = random.gauss(100, 15)
        if self.age < 14: self.intelligence -= (14 - self.age) * 2.5
        elif self.age > 70: self.intelligence -= (self.age - 70) * 0.6

        # BUGFIX #3 : les stats max representent le potentiel SAIN. La maladie est un
        # malus separe (cf. _facteur_maladie) applique sur la valeur courante, afin
        # que la guerison de la maladie pendant recuperation() rende la force/vitesse.
        self.force_max = self.force
        self.vitesse_max = self.vitesse
        facteur_maladie = self._facteur_maladie()
        self.force *= facteur_maladie
        self.vitesse *= facteur_maladie

    def _calculer_entrainement(self):
        self.niveau_entrainement = 0.0
        # LOI DU REEL (entrainement) : avant, 35 % des hommes et 20 % des femmes de 17 a
        # 49 ans etaient entraines, et personne en dehors : 98 % des champions etaient
        # entraines, aucun de moins de 17 ans ni de plus de 49 ans. Desormais :
        #  - frequence realiste d'une pratique de combat (sport de combat, armee,
        #    police) : pic ~14 % des hommes et ~6 % des femmes de 18 a 35 ans ;
        #  - des 8 ans (judo, karate) et jusqu'au grand age (anciens pratiquants), en
        #    perdant le niveau avec les annees ;
        #  - plus frequente chez les riches (club, coach), rare chez les plus pauvres.
        proba_age = Human._interp(self.age, 8, 0.0, 18, 1.0) if self.age < 18 else Human._interp(self.age, 35, 1.0, 75, 0.15)
        proba_sexe = 0.14 if self.sexe == 'Homme' else 0.06
        proba_richesse = Human.ACCES_ENTRAINEMENT[Human.CLASSES_SOCIALES.index(self.classe_sociale)]
        if self.age >= 8 and random.random() < proba_age * proba_sexe * proba_richesse:
            self.niveau_entrainement = random.betavariate(2, 5)  # OPTIMISATION #1
            # Niveau plafonne par l'age : un enfant debute, un ancien a perdu la main.
            if self.age < 18: self.niveau_entrainement *= Human._interp(self.age, 8, 0.3, 18, 1.0)
            elif self.age > 45: self.niveau_entrainement *= Human._interp(self.age, 45, 1.0, 80, 0.4)
            self.resilience = min(1.0, self.resilience + self.niveau_entrainement * 0.3)  # 5.1 : l'entrainement endurcit
            # BUGFIX #3 : le bonus s'ajoute au potentiel SAIN (force_max/vitesse_max),
            # puis on reapplique le malus de maladie sur la valeur courante.
            self.force_max *= 1 + Human.GAIN_FORCE_ENTRAINEMENT * self.niveau_entrainement
            self.vitesse_max *= 1 + Human.GAIN_VITESSE_ENTRAINEMENT * self.niveau_entrainement
            facteur_maladie = self._facteur_maladie()
            self.force = self.force_max * facteur_maladie
            self.vitesse = self.vitesse_max * facteur_maladie

    def _calculer_richesse(self, region_data):
        base_region = region_data['eco_factor']
        if self.age < 18: facteur_age = 0.05
        elif self.age < 25: facteur_age = 0.2 + (self.age - 18) * 0.05
        elif self.age < 60: facteur_age = 0.6 + min(0.4, (self.age - 25) * 0.01)
        else: facteur_age = max(0.4, 1.0 - (self.age - 60) * 0.02)
        # BUGFIX (classe « Fortune » jamais atteinte) : avec un écart-type log de 0.8,
        # dépasser 50 x la référence régionale demandait un tirage à +4,9 sigma
        # (0 cas sur 60 000). Le patrimoine réel est bien plus inégal : sigma = 1.5.
        wealth = base_region * facteur_age * random.lognormvariate(0, 1.5)
        self.richesse = round(wealth, 2)
        if self.richesse < base_region * 0.1: index = 0
        elif self.richesse < base_region * 0.3: index = 1
        elif self.richesse < base_region * 0.5: index = 2
        elif self.richesse < base_region * 0.8: index = 3
        elif self.richesse < base_region * 1.2: index = 4
        elif self.richesse < base_region * 2.0: index = 5
        elif self.richesse < base_region * 4.0: index = 6
        elif self.richesse < base_region * 10.0: index = 7
        elif self.richesse < base_region * 50.0: index = 8
        else: index = 9
        self.classe_sociale = self.CLASSES_SOCIALES[index]
        # LOI DU REEL (richesse) : avant, la richesse ne servait a rien en combat.
        self.acces_soins = index >= Human.INDEX_ACCES_SOINS

    # OPTIMISATION (#5) : construction du corps factorisee a partir de PROPORTIONS_CORPS
    # (plus de 6 ratios reecrits ; partage avec integrer_joueur).
    @staticmethod
    def _construire_corps(age, sexe, taille, poids):
        # LOI DU REEL (corpulence) : les PV suivent la masse STRUCTURELLE (masse maigre
        # + un quart de la masse grasse, qui amortit un peu), plus le poids brut.
        # Avant, chaque kilo de graisse donnait autant de PV qu'un kilo de muscle.
        gras = Human.taux_gras(age, sexe, poids / (taille / 100) ** 2)
        masse_structurelle = poids * (1 - gras) + 0.25 * poids * gras
        base_pv = (20 + 0.6 * masse_structurelle) * Human.robustesse(age)
        return {membre: {'pv_max': round(base_pv * prop), 'pv_actuels': round(base_pv * prop)}
                for membre, prop in Human.PROPORTIONS_CORPS.items()}

    def _initialiser_corps(self):
        self.corps = Human._construire_corps(self.age, self.sexe, self.taille, self.poids)
        self.pv_total = sum(m['pv_actuels'] for m in self.corps.values())
        self.pv_max_total = sum(m['pv_max'] for m in self.corps.values())

    @staticmethod
    def _etat_membre(membre):
        # 0.0 = membre rompu ; de 0.5 (tres abime) a 1.0 (intact) sinon.
        pv = membre['pv_actuels']
        if pv <= 0: return 0.0
        return 0.5 + 0.5 * min(1.0, pv / membre['pv_max'])

    def _actualiser_stats(self):
        """
        BUGFIX (le repos affaiblit) : UNIQUE regle qui derive force/vitesse du potentiel
        sain (force_max/vitesse_max), de l'etat des membres et de la maladie. Elle sert
        pendant le combat (apres chaque coup sur un membre) ET apres le repos. Avant,
        le combat ne penalisait un bras qu'a la rupture (x0.7) alors que recuperation()
        recalculait force = force_max x PV bras / PV max : un vainqueur sortait du repos
        plus faible qu'en fin de combat. Comme le repos ne fait que remonter les PV et
        reculer la maladie, il ne peut plus faire baisser une stat.
        """
        corps = self.corps
        etat = Human._etat_membre
        # Bras : l'autre bras frappe encore -> moyenne de la paire. Deux bras rompus :
        # 30 % (coups de pied, de tete). Jambes : une seule jambe rompue cloue sur
        # place -> minimum de la paire.
        bras = (etat(corps['bras_g']) + etat(corps['bras_d'])) / 2
        jambes = min(etat(corps['jambe_g']), etat(corps['jambe_d']))
        # Maladie (reversible si aigue) et handicap (durable) : malus multiplicatifs.
        facteur_sante = self._facteur_maladie() * (1 - 0.6 * self.handicap)
        self.force = self.force_max * (0.3 + 0.7 * bras) * facteur_sante
        self.vitesse = self.vitesse_max * (0.25 + 0.75 * jambes) * facteur_sante

    def _verifier_capacite_combat(self):
        capable = True
        if self.age < 8: capable = False
        if self.handicap > 0.8: capable = False
        if self.severite_maladie > 0.8: capable = False
        # BUGFIX : un combattant dont la tete ou le torse est detruit ne peut plus
        # se battre. Avant, _verifier_capacite_combat ignorait les PV du corps : un
        # "survivant" a la tete a -50 (issu de la branche double-incapacite de
        # combat_a_mort) restait capable et pouvait avancer dans le tournoi sans
        # jamais avoir combattu reellement.
        corps = getattr(self, 'corps', None)
        if corps is not None and (corps['tete']['pv_actuels'] <= 0 or corps['torse']['pv_actuels'] <= 0):
            capable = False
        self.peut_se_battre = capable

    def cloner(self):
        """Copie independante (meme etat, nouvel id). Bien plus rapide que copy.deepcopy."""
        clone = Human.__new__(Human)
        clone.__dict__.update(self.__dict__)
        clone.corps = {membre: dict(data) for membre, data in self.corps.items()}
        clone._est_joueur = False
        Human._counter += 1
        clone.id = Human._counter
        return clone

    def raison_incapacite(self):
        """Explique pourquoi peut_se_battre est faux (None si le combattant est apte)."""
        if self.peut_se_battre: return None
        if self.age < 8: return "moins de 8 ans"
        if self.handicap > 0.8: return "handicap trop lourd"
        if self.severite_maladie > 0.8: return "maladie trop severe"
        return "tete ou torse detruit"

    def __str__(self):
        return (f"ID:{self.id:03d} {self.prenom:10s} ({self.age}ans, {self.sexe[0]}) | For:{self.force:.0f} Vit:{self.vitesse:.0f} | {self.classe_sociale}")

## --- 2. FONCTIONS DE SIMULATION ---


In [ ]:
# Constantes de ciblage (definies une seule fois, pas a chaque coup).
import itertools
CIBLES = ['tete', 'torse', 'bras_g', 'bras_d', 'jambe_g', 'jambe_d']
POIDS_CIBLES_NORMAL = [0.1, 0.4, 0.1, 0.1, 0.15, 0.15]     # ciblage instinctif
# BUGFIX (la panique visait mieux que le calme) : la panique donnait 20 % de coups a
# la tete, contre 10 % en instinctif, alors que la tete est la zone la plus rentable.
# Un paniqué frappe au hasard dans la masse : moins de tete que l'instinctif, plus de
# membres. Part de tete : panique 6 % < instinctif 10 % < tactique 18 %.
POIDS_CIBLES_PANIQUE = [0.06, 0.30, 0.16, 0.16, 0.16, 0.16]    # mental bas -> coups disperses
# 2.2 : ciblage tactique d'un combattant lucide ET intelligent (vise les zones vitales).
POIDS_CIBLES_TACTIQUE = [0.18, 0.50, 0.06, 0.06, 0.10, 0.10]
CUM_CIBLES_NORMAL = list(itertools.accumulate(POIDS_CIBLES_NORMAL))
CUM_CIBLES_PANIQUE = list(itertools.accumulate(POIDS_CIBLES_PANIQUE))
CUM_CIBLES_TACTIQUE = list(itertools.accumulate(POIDS_CIBLES_TACTIQUE))

# 2.1 : echelle de degats. Les degats bruts (force ~50-90) ecrasaient des PV faibles
# -> un seul coup tuait. On reduit fortement pour des duels a plusieurs echanges ;
# la tete reste la zone la plus efficace, sans etre un one-shot.
DEGATS_SCALE = 0.22
MULT_ZONE = {'tete': 1.5, 'torse': 1.0}   # membres : 0.5 (valeur par defaut)
# BUGFIX (aucun malus de precision sur la tete) : la tete est petite, mobile et
# protegee par la garde ; un membre est plus fin et plus mobile que le torse. Sans ce
# malus, viser seulement la tete gagnait 99 % des duels entre clones.
PRECISION_ZONE = {'tete': 0.50, 'torse': 1.0}   # membres : 0.8 (valeur par defaut)
ESQUIVE_ZONE = {'tete': 1.3}                    # on protege sa tete d'instinct ; autres zones : 1.0
MEMBRES_MOTEURS = frozenset(['bras_g', 'bras_d', 'jambe_g', 'jambe_d'])

# 2.3 : cout d'endurance par action. L'entrainement economise le souffle,
# le surpoids et l'age le font fondre plus vite.
def _cout_endurance(c):
    imc = c.poids / (c.taille / 100) ** 2 if getattr(c, 'taille', 0) else 22.0
    cout = 0.030
    cout *= 1.0 - 0.4 * getattr(c, 'niveau_entrainement', 0.0)
    # LOI DU REEL (corpulence) : le surpoids essouffle des 25 d'IMC, +5 % par point
    # (avant : +3 % par point au-dela de 27).
    if imc > 25: cout *= 1.0 + (imc - 25) * 0.05
    if c.age > 45: cout *= 1.0 + (c.age - 45) * 0.01
    return cout


# LOI DU REEL (combat) : un duel a mains nues ne finit presque jamais par une mort.
# Avant, chaque perdant mourait (tete a 0 ou torse a -20), en 36 secondes en mediane.
# Issues possibles pour le perdant :
#   'KO'              tete a 0 PV, ou coup a la tete qui eteint d'un seul coup
#   'hors de combat'  torse a 0 PV, ou moitie des PV du corps perdus : ne peut plus continuer
#   'etranglement'    saisie reussie : etrangle ou immobilise jusqu'a la perte de connaissance
#   'abandon'         effondrement mental : se rend
#   'mort'            coup final qui depasse largement la limite de la tete ou du torse
#   'forfait'         incapable de se battre avant meme le duel
#   'aux points'      impasse, departagee aux PV restants
CAUSES_ELIMINATION = ['KO', 'hors de combat', 'etranglement', 'abandon', 'mort', 'forfait', 'aux points']
SEUIL_MORT = 0.4            # depassement fatal : PV de la zone sous -40 % de son maximum
SEUIL_HORS_COMBAT = 0.5     # part des PV du corps en dessous de laquelle on ne peut plus continuer
# Duree d'une action (secondes) = TEMPS_ACTION / vitesse : ~3 s a vitesse 50 (les
# adversaires se jaugent, tournent, s'accrochent). Avant : 75, soit un coup toutes
# les 1,5 s sans jamais souffler.
TEMPS_ACTION = 150.0


def _eliminer(perdant, cause):
    # Cause d'elimination du perdant (cf. CAUSES_ELIMINATION). Sert aux statistiques.
    perdant.cause_elimination = cause


def duel(combattant_1, combattant_2, verbose=True):
    """
    Duel a mains nues jusqu'a l'elimination d'un des deux combattants. Renvoie le
    vainqueur ; le perdant recoit `cause_elimination` (cf. CAUSES_ELIMINATION).
    """
    # Verifications pre-combat (cas des incapables).
    if not combattant_1.peut_se_battre and not combattant_2.peut_se_battre:
        # Les DEUX sont incapables : on departe au total de PV, seul le perdant est
        # acheve, le survivant est stabilise au minimum vital puis reverifie.
        pv1 = sum(m['pv_actuels'] for m in combattant_1.corps.values())
        pv2 = sum(m['pv_actuels'] for m in combattant_2.corps.values())
        # BUGFIX (incoherence #4) : a egalite parfaite de PV on tire le vainqueur au
        # sort, au lieu de toujours favoriser combattant_1 (biais d'ordre lie a sa
        # position dans la liste, non neutre meme apres shuffle des paires).
        if pv1 == pv2:
            winner = random.choice([combattant_1, combattant_2])
        else:
            winner = combattant_1 if pv1 > pv2 else combattant_2
        loser = combattant_2 if winner is combattant_1 else combattant_1
        loser.corps['tete']['pv_actuels'] = -50
        _eliminer(loser, 'forfait')
        # On stabilise le survivant au minimum vital (tete/torse).
        if winner.corps['tete']['pv_actuels'] <= 0:
            winner.corps['tete']['pv_actuels'] = 1
        if winner.corps['torse']['pv_actuels'] <= 0:
            winner.corps['torse']['pv_actuels'] = 1
        # BUGFIX (incoherence #2) : on leve aussi les incapacites NON physiques du
        # survivant (handicap du grand age, maladie au-dela du seuil critique). Sans
        # cela, _verifier_capacite_combat le redeclare incapable, il retombe a chaque
        # round dans cette meme branche et "gagne" par defaut sans jamais combattre --
        # au point de pouvoir etre sacre champion sans avoir porte un seul coup. Apres
        # stabilisation, il pourra reellement se battre au round suivant.
        if winner.handicap > 0.8:
            winner.handicap = 0.8
        if winner.severite_maladie > 0.8:
            winner.severite_maladie = 0.8
        winner.pv_total = sum(m['pv_actuels'] for m in winner.corps.values())
        winner._verifier_capacite_combat()
        return winner
    if not combattant_1.peut_se_battre:
        combattant_1.corps['tete']['pv_actuels'] = -50
        _eliminer(combattant_1, 'forfait')
        return combattant_2
    if not combattant_2.peut_se_battre:
        combattant_2.corps['tete']['pv_actuels'] = -50
        _eliminer(combattant_2, 'forfait')
        return combattant_1

    if verbose:
        print(f"--- DUEL : {combattant_1.prenom} (M:{combattant_1.mental:.0%}) VS {combattant_2.prenom} (M:{combattant_2.mental:.0%}) ---")

    timer_c1, timer_c2 = 0.0, 0.0
    winner = None
    # 2.3 : endurance normalisee (1.0 = frais, 0.0 = epuise), indexee par is_c1.
    endurance = {True: 1.0, False: 1.0}

    # Garde-fou anti-boucle-infinie (double tetanie / impasse).
    tours = 0
    MAX_TOURS = 100000

    while True:
        tours += 1
        if tours > MAX_TOURS:
            pv1 = sum(m['pv_actuels'] for m in combattant_1.corps.values())
            pv2 = sum(m['pv_actuels'] for m in combattant_2.corps.values())
            # #4 : tie-break aleatoire en cas d'egalite parfaite (cf. plus haut).
            winner = random.choice([combattant_1, combattant_2]) if pv1 == pv2 else (combattant_1 if pv1 > pv2 else combattant_2)
            if verbose: print("--- IMPASSE : combat departage aux points ---")
            break

        # Si les deux sont tetanises (mental <= 0.1), personne n'attaque -> on departage.
        if combattant_1.mental <= 0.1 and combattant_2.mental <= 0.1:
            pv1 = sum(m['pv_actuels'] for m in combattant_1.corps.values())
            pv2 = sum(m['pv_actuels'] for m in combattant_2.corps.values())
            # #4 : tie-break aleatoire en cas d'egalite parfaite (cf. plus haut).
            winner = random.choice([combattant_1, combattant_2]) if pv1 == pv2 else (combattant_1 if pv1 > pv2 else combattant_2)
            if verbose: print("--- DOUBLE TETANIE : combat departage aux points ---")
            break

        if timer_c1 <= timer_c2:
            attaquant, defenseur = combattant_1, combattant_2
            timer_att = timer_c1
            is_c1 = True
        else:
            attaquant, defenseur = combattant_2, combattant_1
            timer_att = timer_c2
            is_c1 = False

        # Temps d'action de base (plus on est rapide, plus on agit souvent).
        base_time = TEMPS_ACTION / attaquant.vitesse if attaquant.vitesse > 0 else 6.0
        if attaquant.mental < 0.3: base_time *= 1.5   # le stress ralentit

        # 2.3 : mise a jour de la fatigue de l'attaquant.
        end_att = endurance[is_c1]
        end_att = min(1.0, end_att + 0.006)                  # leger souffle entre deux actions
        end_att = max(0.0, end_att - _cout_endurance(attaquant))
        endurance[is_c1] = end_att
        # BUGFIX (incoherence #3) : le defenseur (celui qui n'agit pas ce tour-ci)
        # reprend lui aussi son souffle. Avant, seul l'attaquant regenerait son
        # endurance ; celui qui subissait une longue serie d'attaques s'epuisait donc
        # anormalement, sans jamais pouvoir respirer entre deux coups recus.
        endurance[not is_c1] = min(1.0, endurance[not is_c1] + 0.006)
        facteur_fatigue = 0.4 + 0.6 * end_att                # 1.0 frais -> 0.4 epuise
        base_time *= 1.0 + (1.0 - end_att) * 0.6             # un epuise agit plus lentement

        if is_c1: timer_c1 += base_time
        else: timer_c2 += base_time
        temps_combat = timer_att

        # Crise de panique : mental brise -> passe son tour.
        if attaquant.mental <= 0.1:
            if verbose: print(f"[{temps_combat:.1f}s] {attaquant.prenom} est tetanise par la terreur !")
            continue

        # LOI DU REEL (combat) : SAISIE. L'attaquant tente parfois d'agripper, de
        # projeter et d'etrangler (plus souvent s'il est entraine). La reussite depend
        # de la domination physique (force x racine du rapport de poids) et de l'ecart
        # de technique. C'est ici que le gabarit pese le plus.
        if random.random() < 0.04 + 0.20 * attaquant.niveau_entrainement:
            domination = (attaquant.force / max(1.0, defenseur.force)) * (attaquant.poids / defenseur.poids) ** 0.5
            p_saisie = 0.08 * domination ** 1.5 + 0.35 * (attaquant.niveau_entrainement - defenseur.niveau_entrainement)
            p_saisie *= facteur_fatigue * (0.5 + 0.5 * attaquant.mental)
            endurance[is_c1] = max(0.0, endurance[is_c1] - 0.03)      # lutter epuise
            if random.random() < max(0.0, min(0.6, p_saisie)):
                _eliminer(defenseur, 'etranglement')
                winner = attaquant
                if verbose: print(f"[{temps_combat:.1f}s] {attaquant.prenom} saisit {defenseur.prenom} et l'etrangle jusqu'a l'inconscience.")
                break
            if verbose and random.random() < 0.2:
                print(f"[{temps_combat:.1f}s] {attaquant.prenom} tente une saisie, {defenseur.prenom} se degage.")
            continue

        # 2.2 : choix de la cible. Lucide + intelligent -> ciblage tactique vital ;
        # mental bas -> panique ; sinon instinctif.
        composure = attaquant.mental * min(1.0, attaquant.intelligence / 120.0)
        if composure >= 0.6:
            cum = CUM_CIBLES_TACTIQUE
        elif attaquant.mental < 0.5:
            cum = CUM_CIBLES_PANIQUE
        else:
            cum = CUM_CIBLES_NORMAL
        cible = random.choices(CIBLES, cum_weights=cum, k=1)[0]

        # 2.2 : la PRECISION depend de l'entrainement, du sang-froid et du souffle.
        # BUGFIX (incoherence #1) : l'ecart de vitesse n'est PLUS compte ici. Il est
        # deja valorise deux fois ailleurs : (a) par la cadence d'action
        # (base_time = 75/vitesse -> le plus rapide agit plus souvent) et (b) par
        # l'esquive du defenseur ci-dessous. L'ajouter aussi a la precision revenait
        # a payer TROIS fois le meme avantage de vitesse. On ne le garde donc que sur
        # l'esquive (interaction defensive ou les reflexes relatifs ont du sens).
        precision_base = 0.6 + 0.30 * attaquant.niveau_entrainement
        precision_base *= (0.5 + 0.5 * attaquant.mental)     # malus si mental bas
        precision_base *= (0.6 + 0.4 * facteur_fatigue)      # 2.3 : un bras epuise vise mal
        precision_base *= PRECISION_ZONE.get(cible, 0.8)     # zone petite/mobile = plus dure a toucher
        precision_base = max(0.05, min(0.95, precision_base))

        if random.random() < precision_base:
            # 2.4 : DEFENSE ACTIVE. Le defenseur tente d'esquiver/parer selon ses
            # reflexes (vitesse), son entrainement, son sang-froid et son souffle.
            end_def = endurance[not is_c1]
            esquive = 0.10 + 0.25 * defenseur.niveau_entrainement
            esquive += (defenseur.vitesse - attaquant.vitesse) / 300
            esquive *= (0.4 + 0.6 * defenseur.mental)
            esquive *= (0.4 + 0.6 * (0.4 + 0.6 * end_def))
            esquive *= ESQUIVE_ZONE.get(cible, 1.0)
            esquive = max(0.0, min(0.75, esquive))
            if random.random() < esquive:
                endurance[not is_c1] = max(0.0, end_def - 0.02)   # esquiver coute de l'energie
                if verbose and random.random() < 0.2:
                    print(f"[{temps_combat:.1f}s] {defenseur.prenom} esquive l'attaque de {attaquant.prenom}.")
                continue

            # 2.1 : degats fortement reduits (DEGATS_SCALE) + modulation par la fatigue.
            degats = attaquant.force * random.uniform(0.8, 1.2) * DEGATS_SCALE
            degats *= MULT_ZONE.get(cible, 0.5)
            degats *= facteur_fatigue
            degats_finaux = max(1, round(degats))

            membre = defenseur.corps[cible]
            etait_deja_casse = membre['pv_actuels'] <= 0
            membre['pv_actuels'] -= degats_finaux

            # BUGFIX (le repos affaiblit) : un membre touche degrade tout de suite la
            # force/vitesse, par la meme regle que celle appliquee apres le repos.
            if cible in MEMBRES_MOTEURS:
                defenseur._actualiser_stats()

            if verbose:
                print(f"[{temps_combat:.1f}s] {attaquant.prenom} frappe {defenseur.prenom} au {cible} ({degats_finaux} dmg).")

            # Impact psychologique (5.1 : module par la resilience individuelle).
            res_def = getattr(defenseur, 'resilience', 0.5)
            defenseur.mental -= 0.02 * (degats_finaux / 10) * (1.3 - res_def)   # la douleur traumatise
            defenseur.mental = max(0, defenseur.mental)

            res_att = getattr(attaquant, 'resilience', 0.5)
            if attaquant.niveau_entrainement < 0.3:
                attaquant.mental -= 0.01 * (1.3 - res_att)     # les non-aguerris sont choques
            else:
                attaquant.mental -= 0.002 * (1.3 - res_att)    # les guerriers sont endurcis
            attaquant.mental = max(0, attaquant.mental)

            # Effets a la rupture du membre (transition >0 -> <=0).
            if membre['pv_actuels'] <= 0 and not etait_deja_casse:
                defenseur.mental -= 0.15 * (1.3 - res_def)     # choc de la blessure grave
                defenseur.mental = max(0, defenseur.mental)
                defenseur._trauma_profond = True               # 4.3 : cicatrice psy durable
                if verbose: print(f"   >> {defenseur.prenom} est gravement blesse ! Panique...")

            # LOI DU REEL (combat) : fin du duel. Le perdant est le plus souvent mis
            # KO ou hors de combat ; il ne meurt que si le coup final depasse largement
            # la limite de la zone (victime fragile, frappeur tres puissant).
            cause = None
            if cible in ('tete', 'torse'):
                pv_zone, pv_max_zone = membre['pv_actuels'], membre['pv_max']
                if pv_zone <= -SEUIL_MORT * pv_max_zone:
                    cause = 'mort'
                elif pv_zone <= 0:
                    cause = 'KO' if cible == 'tete' else 'hors de combat'
                elif cible == 'tete':
                    # KO sur un seul coup : d'autant plus probable que le coup est lourd
                    # pour cette tete et qu'elle a deja encaisse.
                    deja_encaisse = 1 - pv_zone / pv_max_zone
                    p_ko = (degats_finaux / pv_max_zone - 0.15) + 0.3 * deja_encaisse
                    if random.random() < min(0.7, p_ko):
                        cause = 'KO'
            if cause is None:
                # Usure generale : la moitie des PV du corps perdus (membres rompus
                # comptes a 0), on ne tient plus debout.
                pv_restants = sum(max(0, m['pv_actuels']) for m in defenseur.corps.values())
                if pv_restants <= SEUIL_HORS_COMBAT * defenseur.pv_max_total:
                    cause = 'hors de combat'
            if cause is None and defenseur.mental < 0.4:
                # Abandon : certain a mental 0, de plus en plus probable sous 0.4.
                if defenseur.mental <= 0 or random.random() < (0.4 - defenseur.mental) * 1.5:
                    cause = 'abandon'

            if cause is not None:
                _eliminer(defenseur, cause)
                winner = attaquant
                if cause == 'mort':
                    winner.mental = max(0, winner.mental - 0.1 * (1.3 - res_att))
                    winner._trauma_profond = True                  # 4.3 : tuer laisse une trace
                else:
                    winner.mental = max(0, winner.mental - 0.03 * (1.3 - res_att))
                if verbose:
                    if cause == 'mort': print(f"--- MORT DE {defenseur.prenom} ---")
                    elif cause == 'abandon': print(f"--- {defenseur.prenom} s'effondre mentalement et abandonne ---")
                    else: print(f"--- {defenseur.prenom} est {cause} ---")
                break

        else:
            if verbose and random.random() < 0.2:
                print(f"[{temps_combat:.1f}s] {attaquant.prenom} manque son coup.")

    # Post-combat.
    if winner:
        loser = combattant_2 if winner is combattant_1 else combattant_1
        if getattr(loser, 'cause_elimination', None) is None:
            _eliminer(loser, 'aux points')    # impasse ou double tetanie
        winner.pv_total = sum(m['pv_actuels'] for m in winner.corps.values())
        winner.mental = min(winner.mental_max, winner.mental + 0.05)   # leger apaisement
    return winner


# Ancien nom, garde pour compatibilite : le duel ne se termine plus forcement par une mort.
combat_a_mort = duel


In [36]:
# Taux de recuperation mentale (points/heure).
TAUX_MENTAL_NATUREL = 0.01
TAUX_MENTAL_MEDICAL = 0.05
# Maladie : etat reversible (taux de guerison par heure).
TAUX_MALADIE_NATUREL = 0.005
TAUX_MALADIE_MEDICAL = 0.02


def recuperation(individu, temps_heures, equipe_medecin):
    """
    Recuperation entre deux rounds. 24h de repos restaurent surtout la CONDITION
    physique et le moral ; les blessures profondes (membres detruits) ne se
    "reparent" pas en une journee.
    """
    # LOI DU REEL (richesse) : un individu aise se paie des soins meme sans equipe
    # medicale fournie par le tournoi.
    equipe_medecin = equipe_medecin or getattr(individu, 'acces_soins', False)
    taux_mental_reel = TAUX_MENTAL_MEDICAL if equipe_medecin else TAUX_MENTAL_NATUREL

    # La douleur ralentit la recuperation mentale (ratio borne dans [0, 1]).
    pv_max_total = sum(m['pv_max'] for m in individu.corps.values())
    pv_actuel_total = sum(m['pv_actuels'] for m in individu.corps.values())
    ratio_sante = max(0.0, min(1.0, pv_actuel_total / pv_max_total)) if pv_max_total > 0 else 1.0

    # --- Guerison du corps ---
    for membre, data in individu.corps.items():
        pv_actuels = data['pv_actuels']
        pv_max = data['pv_max']
        if pv_actuels < pv_max:
            if pv_actuels <= 0:
                # 4.2 : fracture / membre detruit -> PAS de reparation en 24h. Avec une
                # equipe medicale on stabilise (anti-infection) et on amorce une
                # consolidation LENTE (~2 PV/jour) : le membre reste hors d'usage
                # plusieurs rounds avant de redevenir fonctionnel. Sans soins, quasi nul.
                if equipe_medecin:
                    data['pv_actuels'] = min(pv_max, pv_actuels + 2.0 * (temps_heures / 24.0))
                else:
                    data['pv_actuels'] = min(pv_max, pv_actuels + 0.2 * (temps_heures / 24.0))
            else:
                # 4.1 : blessure legere (PV>0). Le repos restaure surtout la CONDITION
                # (fatigue, contusions, coupures superficielles), pas une cicatrisation
                # lente en PV/h -> recuperation proportionnelle au deficit.
                deficit = pv_max - pv_actuels
                taux_condition = 0.6 if equipe_medecin else 0.30   # fraction du deficit par 24h
                data['pv_actuels'] = min(pv_max, pv_actuels + deficit * taux_condition * (temps_heures / 24.0))

    # --- 4.3 : Guerison mentale NON lineaire, liee a la resilience et au trauma ---
    resilience = getattr(individu, 'resilience', 0.5)
    max_mental = getattr(individu, 'mental_max', 1.0)
    # Cicatrice psychologique (PTSD) : un trauma profond abaisse durablement le
    # plafond mental -> la recuperation plafonne sous le niveau initial.
    if getattr(individu, '_trauma_profond', False):
        individu.mental_max = max(0.3, max_mental - 0.05 * (1 - resilience))
        max_mental = individu.mental_max
        individu._trauma_profond = False
    # Recuperation rapide loin du plafond, de plus en plus lente en l'approchant ;
    # modulee par la resilience individuelle et par la douleur (ratio_sante).
    ecart = max(0.0, max_mental - individu.mental)
    taux_mental_eff = taux_mental_reel * (0.5 + resilience)
    recup_mentale = temps_heures * taux_mental_eff * ratio_sante * (0.3 + 0.7 * ecart)
    individu.mental = min(max_mental, individu.mental + recup_mentale)

    # --- Guerison de la maladie (etat reversible) ---
    if getattr(individu, 'est_malade', False):
        taux_maladie = TAUX_MALADIE_MEDICAL if equipe_medecin else TAUX_MALADIE_NATUREL
        if individu.maladie_chronique:
            # LOI DU REEL (sante) : une maladie chronique ne guerit pas. Sans medecin,
            # rien ne bouge ; avec, un traitement la soulage jusqu'a un plancher.
            if equipe_medecin:
                individu.severite_maladie = max(individu._severite_plancher, individu.severite_maladie - taux_maladie * temps_heures)
        else:
            individu.severite_maladie = max(0.0, individu.severite_maladie - taux_maladie * temps_heures)
            if individu.severite_maladie <= 0.05:
                individu.est_malade = False
                individu.severite_maladie = 0.0

    # --- Recalcul des stats derivees (force/vitesse) selon l'etat des membres ---
    # BUGFIX (le repos affaiblit) : meme regle que pendant le combat
    # (Human._actualiser_stats), donc le repos ne peut que rendre de la force.
    individu._actualiser_stats()

    individu.pv_total = sum(m['pv_actuels'] for m in individu.corps.values())
    individu._verifier_capacite_combat()


## --- 3. FONCTION HUNGER GAMES (Logique du tournoi) ---

In [ ]:
import math


def creer_population(taille, seed=None):
    """Genere `taille` humains. Avec une graine (seed), la population est reproductible."""
    if seed is not None:
        random.seed(seed)
    return [Human() for _ in range(taille)]


def _round_tableau(population, round_num, equipe_medecin, signaler, verbose):
    """Round de tableau classique : on apparie la population, elle est divisee par deux."""
    survivors = []
    random.shuffle(population)

    for i in range(0, len(population), 2):
        if i + 1 >= len(population):
            # FILET DE SECURITE (incoherence #5) : gestion du combattant exempte
            # quand la population est IMPAIRE. Avec une taille en puissance de 2
            # (8192), chaque round divise exactement par deux et cette branche reste
            # inactive : ce n'est donc PAS du code mort mais un garde-fou pour toute
            # autre taille de population (ex. si on change TAILLE_POPULATION).
            # L'exempte recupere aussi 24h pour ne pas etre desavantage face aux
            # vainqueurs de duel, et l'exemption est desormais signalee (plus de
            # bye silencieux qui passait inapercu).
            if verbose:
                print(f"   (Round {round_num} : population impaire -> {population[i].prenom} est exempte ce round)")
            recuperation(population[i], 24, equipe_medecin)
            survivors.append(population[i])
            break

        h1 = population[i]
        h2 = population[i+1]

        # DUEL
        winner = duel(h1, h2, verbose=False)

        # RECUPERATION : 24h (1 jour)
        recuperation(winner, 24, equipe_medecin)

        survivors.append(winner)

        # --- VERIFICATION DES JOUEURS ET AFFICHAGE COMPARATIF ---
        signaler(h2 if winner is h1 else h1, winner, round_num)

    return survivors


def _round_echantillonne(population, tracked, round_num, equipe_medecin, signaler):
    """
    Round ou l'humanite en vie depasse l'echantillon simule : l'echantillon garde sa
    taille. Chaque membre affronte la COPIE d'un autre membre tire au hasard (son
    adversaire reel, hors echantillon, suit la meme loi) et le vainqueur occupe la
    place. Un joueur suivi ne combat qu'une fois par round et n'est jamais copie.
    """
    suivis = [h for h in population if h.id in tracked]
    autres = [h for h in population if h.id not in tracked]
    random.shuffle(autres)
    k = len(autres)
    # Toutes les copies sont prises AVANT le premier combat : elles portent l'etat
    # d'entree de round, pas un etat deja abime par un duel de ce round.
    adversaires_suivis = [random.choice(autres).cloner() for _ in suivis]
    copies = [h.cloner() for h in autres]
    # Decalage circulaire : chaque membre sert exactement deux fois (une fois en
    # personne, une fois en copie) et ne rencontre jamais sa propre copie.
    decalage = random.randrange(1, k)

    survivors = []
    for joueur, adversaire in zip(suivis, adversaires_suivis):
        winner = duel(joueur, adversaire, verbose=False)
        recuperation(winner, 24, equipe_medecin)
        survivors.append(winner)
        if winner is not joueur:
            signaler(joueur, winner, round_num)
    for i in range(k):
        winner = duel(autres[i], copies[(i + decalage) % k], verbose=False)
        recuperation(winner, 24, equipe_medecin)
        survivors.append(winner)
    return survivors


def hunger_games(population, tracked_ids=None, equipe_medecin=False, nb_rounds=None,
                 seed=None, niveaux=None, adversaires_par_round=512, verbose=True):
    """
    Tournoi a elimination directe. Renvoie le champion.

    nb_rounds=None : tableau classique sur `population` seule (8192 -> 13 rounds).

    nb_rounds=33   : l'humanite compte 2^33 personnes (8,6 milliards) et `population`
    en est un ECHANTILLON. BUGFIX (13 rounds, pas 33) : avant, 8192 = 2^13 humains ne
    donnaient que 13 rounds et les rounds 14 a 33 n'existaient pas. Impossible de creer
    8,6 milliards d'objets, donc :
      - tant que l'humanite en vie depasse l'echantillon (rounds 1 a 20 pour 8192),
        l'echantillon garde sa taille : chacun affronte la copie d'un autre membre
        (reechantillonnage, cf. _round_echantillonne) ;
      - ensuite (rounds 21 a 33) l'echantillon EST l'humanite restante : tableau exact.
    Limite assumee : l'echantillon ne contient pas d'individus plus rares que 1 sur
    sa taille ; les adversaires des derniers rounds sont donc les meilleurs de
    l'echantillon, pas les meilleurs de 8,6 milliards.

    seed    : graine du generateur aleatoire (tournoi reproductible).
    niveaux : liste a remplir ; recoit pour chaque round une copie d'au plus
              `adversaires_par_round` combattants entrant dans ce round (sert a
              estimer_survie).
    """
    # BUGFIX : equipe_medecin est desormais propage a recuperation(). Avant, le
    # tournoi appelait toujours recuperation(..., False), rendant toute la branche
    # medicale de recuperation() (taux_medical, stabilisation, platre) inatteignable.
    if tracked_ids is None:
        tracked_ids = []
    # BUGFIX : hunger_games([]) levait un IndexError obscur sur population[0].
    if not population:
        raise ValueError("population vide : il faut au moins un humain.")
    if seed is not None:
        random.seed(seed)

    population = list(population)   # on ne melange pas la liste de l'appelant
    tracked = set(tracked_ids)
    nb_suivis = sum(1 for h in population if h.id in tracked)
    if nb_rounds is not None:
        if 2 ** nb_rounds < len(population):
            raise ValueError(f"2^{nb_rounds} humains ne suffisent pas pour une population de {len(population)}.")
        if 2 ** nb_rounds > len(population) and len(population) - nb_suivis < 2:
            raise ValueError("echantillon trop petit : il faut au moins 2 humains non suivis.")

    historique_joueurs = {pid: "En vie" for pid in tracked_ids}

    def signaler(perdant, gagnant, round_num):
        if perdant.id in tracked:
            historique_joueurs[perdant.id] = f"Éliminé au round {round_num} par {gagnant.prenom} ({perdant.cause_elimination})"
            if verbose:
                print(f"\n>>> LE JOUEUR (ID:{perdant.id}) A ÉTÉ ÉLIMINÉ AU ROUND {round_num} <<<")
                print(f"   PERDANT : {perdant}") # Affiche les stats du joueur
                print(f"   GAGNANT : {gagnant}") # Affiche les stats du tueur pour comparaison
                print(f"   ISSUE   : {perdant.cause_elimination}")

    round_num = 1
    while True:
        if nb_rounds is None:
            if len(population) <= 1: break
            vivants = len(population)
        else:
            if round_num > nb_rounds: break
            vivants = 2 ** (nb_rounds - round_num + 1)
        echantillonne = vivants > len(population)
        if len(population) > vivants:
            # Echantillon dont la taille n'est pas une puissance de 2 : on le ramene a
            # l'humanite restante (les joueurs suivis sont gardes en priorite).
            random.shuffle(population)
            population.sort(key=lambda h: h.id not in tracked)
            population = population[:vivants]

        if niveaux is not None:
            candidats = [h for h in population if h.id not in tracked]
            pris = random.sample(candidats, min(adversaires_par_round, len(candidats)))
            niveaux.append([h.cloner() for h in pris])

        if verbose:
            if echantillonne:
                print(f"\n=== ROUND {round_num} | Humains en vie : {vivants:,} | Échantillon simulé : {len(population)} ===".replace(',', ' '))
            else:
                print(f"\n=== ROUND {round_num} | Survivants : {len(population)} ===")

        if echantillonne:
            population = _round_echantillonne(population, tracked, round_num, equipe_medecin, signaler)
        else:
            population = _round_tableau(population, round_num, equipe_medecin, signaler, verbose)

        # --- Affichage Graphiques ---
        # BUGFIX : l'ancienne condition (== 20 / >= 30) n'etait jamais atteinte :
        # 8192 = 2^13 -> le tournoi dure 13 rounds. On affiche desormais le round 1
        # puis tous les 5 rounds (rounds reellement atteints).
        if verbose and (round_num == 1 or round_num % 5 == 0):
            print(f"--- ANALYSE GRAPHIQUE DU ROUND {round_num} ---")
            # Décommentez les graphiques souhaités
            # graph_parite(population, round_num)
            # graph_force_vitesse(population, round_num)
            # graph_blessures(population, round_num)

        round_num += 1

    # --- BILAN FINAL ---
    champion = population[0]

    if verbose:
        print(f"\n{'='*40}")
        print(f"LE VAINQUEUR EST : {champion}")
        print(f"{'='*40}")

        print(f"\n--- BILAN DES JOUEURS ---")
        for pid, status in historique_joueurs.items():
            if champion.id == pid:
                print(f"Joueur {pid} : VAINQUEUR DU TOURNOI !")
            else:
                print(f"Joueur {pid} : {status}")

    return champion


## 4 Génération des graphique

In [ ]:
def graph_parite(population, round_num):
    """
    Affiche un camembert de la répartition H/F.
    """
    hommes = sum(1 for h in population if h.sexe == 'Homme')
    femmes = len(population) - hommes

    labels = ['Hommes', 'Femmes']
    sizes = [hommes, femmes]
    colors = ['lightskyblue', 'lightcoral']

    plt.figure(figsize=(6, 6))
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    plt.title(f"% d'hommes et de femmes au round {round_num}")
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure (évite l'accumulation en mémoire si appelé en boucle)

In [ ]:
def graph_entrainement(population, round_num):
    """
    Affiche la distribution du niveau d'entraînement (skill) des survivants.
    """
    niveaux = [h.niveau_entrainement for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(niveaux, bins=20, color='green', alpha=0.7, edgecolor='black')
    plt.xlabel("Niveau d'entraînement (0 = Débutant, 1 = Expert)")
    plt.ylabel("Nombre d'individus")
    plt.title(f"Compétence de combat au round {round_num}")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_age(population, round_num):
    """
    Affiche un histogramme des âges.
    """
    ages = [h.age for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(ages, bins=range(0, 110, 5), color='purple', alpha=0.7, edgecolor='black', rwidth=0.85)
    plt.xlabel("Âge")
    plt.ylabel("Effectif")
    plt.title(f"Pyramide des âges au round {round_num}")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_blessures(population, round_num):
    """
    Affiche l'état de santé moyen basé sur le % de PV restants.
    """
    etats = []
    for h in population:
        pv_max_total = sum(m['pv_max'] for m in h.corps.values())
        pv_actuel_total = sum(m['pv_actuels'] for m in h.corps.values())
        pourcentage = (pv_actuel_total / pv_max_total) * 100 if pv_max_total > 0 else 0
        etats.append(pourcentage)

    plt.figure(figsize=(8, 5))
    plt.hist(etats, bins=20, color='red', alpha=0.7, edgecolor='black')
    plt.xlabel("Pourcentage de vie restante (%)")
    plt.ylabel("Nombre de survivants")
    plt.title(f"État de santé de la population au round {round_num}")
    plt.axvline(x=50, color='black', linestyle='--', label='Seuil critique (50%)')
    plt.legend()
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_force_vitesse(population, round_num):
    forces = [h.force for h in population]
    vitesses = [h.vitesse for h in population]

    plt.figure(figsize=(8, 6))
    plt.scatter(forces, vitesses, alpha=0.6, c='blue', edgecolors='w', linewidth=0.5)
    plt.xlabel("Force")
    plt.ylabel("Vitesse")
    plt.title(f"Profil de combat (Force vs Vitesse) au round {round_num}")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_etat_mental(population, round_num):
    mentaux = [h.mental for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(mentaux, bins=20, color='indigo', alpha=0.7, edgecolor='black')
    plt.xlabel("Santé Mentale (0 = Brisé, 1 = Stable)")
    plt.ylabel("Nombre d'individus")
    plt.title(f"État psychologique au round {round_num}")
    plt.axvline(x=0.3, color='red', linestyle='--', label='Seuil de panique')
    plt.legend()
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
from collections import Counter

def graph_regions(population, round_num):
    # OPTIMISATION : un seul passage avec Counter au lieu de regions.count(r)
    # dans une boucle (qui était en O(n^2)).
    compte = Counter(h.region_key for h in population)
    unique_regions = list(compte.keys())
    counts = [compte[r] for r in unique_regions]

    plt.figure(figsize=(10, 5))
    plt.bar(unique_regions, counts, color='teal', edgecolor='black')
    plt.xlabel("Région d'origine")
    plt.ylabel("Nombre de survivants")
    plt.title(f"Survivants par région au round {round_num}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_richesse(population, round_num):
    richesses = [h.richesse for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(richesses, bins=50, color='gold', alpha=0.7, edgecolor='black')
    plt.xlabel("Richesse ($)")
    plt.ylabel("Nombre de survivants")
    plt.title(f"Distribution des richesses au round {round_num}")
    plt.yscale('log') # Souvent utile car les riches sont très riches
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

## 5 Tournoi avec moi

In [ ]:
import statistics

# BUGFIX (entrees non validees) : ecritures acceptees pour le sexe. Avant,
# sexe='homme' passait tel quel et le joueur n'etait ni 'Homme' ni 'Femme'.
SEXES_ACCEPTES = {
    'homme': 'Homme', 'h': 'Homme', 'm': 'Homme', 'masculin': 'Homme', 'male': 'Homme',
    'femme': 'Femme', 'f': 'Femme', 'feminin': 'Femme', 'féminin': 'Femme', 'female': 'Femme',
}


def _valider_nombre(nom, valeur, mini, maxi, aide=""):
    if isinstance(valeur, bool) or not isinstance(valeur, (int, float)) or valeur != valeur:
        raise ValueError(f"{nom} doit être un nombre (reçu : {valeur!r}).")
    if not (mini <= valeur <= maxi):
        raise ValueError(f"{nom}={valeur!r} hors limites : attendu entre {mini} et {maxi}.{aide}")


def creer_joueur(prenom, age, sexe, taille, poids, mental, niveau_entrainement, intelligence=100, handicap=0.0):
    """
    Crée un joueur personnalisé en définissant son physique (Taille/Poids), sans
    l'insérer dans une population. La Force et la Vitesse sont calculées
    automatiquement, sans aucun tirage aléatoire : mêmes entrées -> même joueur.
    """
    # BUGFIX : validation des entrees. Des valeurs <= 0 (taille/poids) produiraient
    # des PV et une force negatifs (combattant "invincible" ou degats qui soignent).
    # BUGFIX (entrees non validees) : taille=1.80 (metres au lieu de centimetres) et
    # sexe='homme' passaient sans erreur. Chaque champ a maintenant ses bornes.
    if not isinstance(prenom, str) or not prenom.strip():
        raise ValueError("prenom doit être un texte non vide.")
    if not isinstance(sexe, str) or sexe.strip().lower() not in SEXES_ACCEPTES:
        raise ValueError(f"sexe={sexe!r} inconnu : attendu 'Homme' ou 'Femme'.")
    sexe = SEXES_ACCEPTES[sexe.strip().lower()]
    _valider_nombre("taille", taille, 50, 260, " La taille est en centimètres (ex. 180, pas 1.80).")
    _valider_nombre("poids", poids, 3, 400, " Le poids est en kilogrammes.")
    _valider_nombre("age", age, 0, 120)
    _valider_nombre("mental", mental, 0.0, 1.0)
    _valider_nombre("niveau_entrainement", niveau_entrainement, 0.0, 1.0)
    _valider_nombre("intelligence", intelligence, 40, 200)
    _valider_nombre("handicap", handicap, 0.0, 1.0)

    # 1. Création de la structure de base
    # OPTIMISATION : on alloue l'objet via __new__ sans passer par __init__, qui
    # générerait un humain aléatoire complet (âge, physique, santé, richesse...)
    # dont ~90% des champs seraient ensuite écrasés. Cela évite des dizaines de
    # tirages aléatoires inutiles et corrige au passage la région/richesse/classe
    # incohérentes qui étaient héritées du Human() aléatoire de base.
    joueur = Human.__new__(Human)
    Human._counter += 1
    joueur.id = Human._counter
    joueur.region_key = 'Joueur'
    joueur.richesse = 0.0
    joueur.classe_sociale = 'Joueur'
    joueur.acces_soins = False      # soins : passer equipe_medecin=True au tournoi

    # 2. Identité et Physique imposés
    joueur.prenom = prenom.strip()
    joueur.age = age
    joueur.sexe = sexe
    joueur.taille = taille
    joueur.poids = poids

    # On neutralise les stats parasites (maladie / grand âge) pour qu'un joueur
    # ne soit jamais déclaré incapable de se battre à tort.
    joueur.est_malade = False
    joueur.severite_maladie = 0.0
    joueur.maladie_chronique = False
    joueur._severite_plancher = 0.0
    # Handicap durable : 0.0 par défaut, à renseigner si le joueur en a un (0 à 1).
    joueur.handicap = handicap

    # 3. Calcul automatique des Stats de combat (basé sur le modèle Human)
    # BUGFIX (joueur tire aux des) : plus AUCUN tirage aleatoire ici. Avant, un
    # gauss(30, 10) etait retire a chaque appel : la force du meme joueur variait de
    # 60 a 93 d'une execution a l'autre. Le joueur recoit la valeur ATTENDUE de son
    # gabarit (Human.stats_attendues), qui tient aussi compte du poids pour la vitesse.
    joueur.force, joueur.vitesse = Human.stats_attendues(joueur.age, joueur.sexe, joueur.taille, joueur.poids)

    # 4. Mental et Compétences
    joueur.mental = mental
    joueur.mental_max = mental
    joueur.niveau_entrainement = niveau_entrainement
    joueur.resilience = max(0.05, min(1.0, 0.4 + 0.4 * niveau_entrainement + 0.2 * mental))  # 5.1
    joueur.intelligence = intelligence

    # Application du bonus d'entraînement (comme dans la classe Human)
    joueur.force *= 1 + Human.GAIN_FORCE_ENTRAINEMENT * joueur.niveau_entrainement
    joueur.vitesse *= 1 + Human.GAIN_VITESSE_ENTRAINEMENT * joueur.niveau_entrainement
    # Sauvegarde des maxs
    joueur.force_max = joueur.force
    joueur.vitesse_max = joueur.vitesse

    # 5. Recalcul des PV du corps en fonction du NOUVEAU poids
    # (Sinon on aurait les PV d'un humain random)
    # OPTIMISATION #5 : meme construction factorisee que Human._initialiser_corps.
    joueur.corps = Human._construire_corps(joueur.age, joueur.sexe, joueur.taille, joueur.poids)
    joueur.pv_total = sum(m['pv_actuels'] for m in joueur.corps.values())
    joueur.pv_max_total = sum(m['pv_max'] for m in joueur.corps.values())

    # Marqueur pour ne jamais écraser un joueur déjà inséré
    joueur._est_joueur = True

    # Vérification finale
    joueur._actualiser_stats()
    joueur._verifier_capacite_combat()
    return joueur


def integrer_joueur(population, prenom, age, sexe, taille, poids, mental, niveau_entrainement, intelligence=100, handicap=0.0):
    """
    Crée un joueur personnalisé (cf. creer_joueur) et l'insère dans la population.
    Renvoie son id.
    """
    joueur = creer_joueur(prenom, age, sexe, taille, poids, mental, niveau_entrainement, intelligence, handicap)

    # 6. Insertion aléatoire
    # BUGFIX : on évite d'écraser un emplacement déjà occupé par un autre joueur.
    # Avant, deux appels successifs pouvaient tirer le même index et le 2e joueur
    # écrasait le 1er -> un seul joueur réel dans le tournoi.
    candidats = [i for i, h in enumerate(population) if not getattr(h, '_est_joueur', False)]
    # BUGFIX : garde-fou si tous les emplacements sont déjà des joueurs
    # (random.choice([]) lèverait un IndexError peu explicite).
    if not candidats:
        raise ValueError("Aucun emplacement libre dans la population pour insérer un joueur.")
    index_aleatoire = random.choice(candidats)
    population[index_aleatoire] = joueur

    print(f">>> JOUEUR AJOUTÉ : {joueur.prenom} (ID:{joueur.id}) | {taille}cm, {poids}kg | For calculée:{joueur.force:.0f} | Vit calculée:{joueur.vitesse:.0f}")
    # BUGFIX (entrees non validees) : age=5 rendait le joueur incapable sans rien dire.
    if not joueur.peut_se_battre:
        print(f"   ATTENTION : {joueur.prenom} ne peut pas se battre ({joueur.raison_incapacite()}). Il perdra son premier duel sans combattre.")

    return joueur.id


def estimer_survie(niveaux, nb_tournois=1000, seed=None, equipe_medecin=False, verbose=True, **profil):
    """
    BUGFIX (un seul tirage, sans graine) : un tournoi unique ne dit rien, le même
    joueur sort au round 1 comme au dernier selon le tirage. On rejoue donc le
    parcours du joueur `nb_tournois` fois et on rend la DISTRIBUTION de son round
    de sortie.

    niveaux : liste remplie par hunger_games(..., niveaux=[...]) ; niveaux[r-1]
              contient des combattants entrant dans le round r. À chaque round, le
              joueur affronte la copie d'un de ces combattants, tirée au hasard.
    profil  : mêmes arguments que creer_joueur (prenom, age, sexe, taille, ...).
    Renvoie un dictionnaire : sorties (round de sortie de chaque tournoi,
    nb_rounds + 1 = champion), mediane, quartiles, p_champion, survie[r-1] =
    probabilité de passer le round r.
    """
    if not niveaux:
        raise ValueError("niveaux est vide : lancez d'abord hunger_games(..., niveaux=niveaux).")
    if nb_tournois < 1:
        raise ValueError("nb_tournois doit être au moins 1.")
    if seed is not None:
        random.seed(seed)

    nb_rounds = len(niveaux)
    sorties = []
    for _ in range(nb_tournois):
        joueur = creer_joueur(**profil)
        sortie = nb_rounds + 1          # nb_rounds + 1 = champion
        for round_num, niveau in enumerate(niveaux, start=1):
            adversaire = random.choice(niveau).cloner()
            if duel(joueur, adversaire, verbose=False) is not joueur:
                sortie = round_num
                break
            recuperation(joueur, 24, equipe_medecin)
        sorties.append(sortie)

    sorties.sort()
    q1, mediane, q3 = (sorties[int(q * (nb_tournois - 1))] for q in (0.25, 0.5, 0.75))
    resultat = {
        'prenom': joueur.prenom,
        'nb_tournois': nb_tournois,
        'nb_rounds': nb_rounds,
        'sorties': sorties,
        'mediane': mediane,
        'quartiles': (q1, q3),
        'p_champion': sum(1 for s in sorties if s > nb_rounds) / nb_tournois,
        'survie': [sum(1 for s in sorties if s > r) / nb_tournois for r in range(1, nb_rounds + 1)],
    }
    if verbose:
        def nom(r): return "le titre" if r > nb_rounds else f"le round {r}"
        print(f"{joueur.prenom} sur {nb_tournois} tournois : sortie médiane = {nom(mediane)} ; "
              f"la moitié des tournois finit entre {nom(q1)} et {nom(q3)} ; "
              f"champion {resultat['p_champion']:.1%} du temps.")
        jalons = [r for r in (1, 2, 3, 5, 8, 13, 21, nb_rounds) if r <= nb_rounds]
        print("   Passe " + " | ".join(f"R{r} : {resultat['survie'][r-1]:.1%}" for r in sorted(set(jalons))))
    return resultat


def graph_survie(*resultats):
    """Courbe de survie (probabilité de passer chaque round) d'un ou plusieurs joueurs."""
    plt.figure(figsize=(9, 5))
    for res in resultats:
        rounds = range(1, res['nb_rounds'] + 1)
        plt.step(rounds, res['survie'], where='post', label=f"{res['prenom']} ({res['nb_tournois']} tournois)")
    plt.xlabel("Round")
    plt.ylabel("Probabilité d'avoir passé ce round")
    plt.title("Jusqu'où va le joueur ? (Monte-Carlo)")
    plt.yscale('log')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend()
    plt.show()
    plt.close()


In [ ]:
if __name__ == "__main__":
    # BUGFIX (13 rounds, pas 33) : 2^33 = 8 589 934 592 humains. On simule un
    # échantillon de 8192 d'entre eux sur les 33 rounds (cf. hunger_games).
    NB_ROUNDS = 33
    TAILLE_POPULATION = 8192
    # BUGFIX (sans graine) : même graine -> même population, même tournoi.
    # Mettre SEED = None pour un tirage différent à chaque exécution.
    SEED = 33

    print(f"--- Génération d'une population ({TAILLE_POPULATION} Humains, échantillon de 2^{NB_ROUNDS}) ---")
    population = creer_population(TAILLE_POPULATION, seed=SEED)

    print(f"Population générée : {len(population)} individus.")

    # --- INTEGRATION DES 2 JOUEURS ---
    # Les profils sont définis une seule fois : ils servent au tournoi ci-dessous
    # et à l'estimation Monte-Carlo de la cellule suivante.

    # JOUEUR 1 : Un "Colosse" (Lourd, Grand -> Fort mais lent)
    PIERRE = dict(
        prenom="Pierre",
        age=26,
        sexe="Homme",
        taille=190,       # Très grand
        poids=94,        # Très lourd -> Force énorme calculée
        mental=0.7,
        niveau_entrainement=0.5,
        intelligence=110
    )

    # JOUEUR 2 : Un "Vif" (Léger, Petit -> Faible mais rapide)
    ALICE = dict(
        prenom="Alice",
        age=24,
        sexe="Femme",
        taille=157,       # Taille moyenne
        poids=49,         # Léger -> Force faible calculée, mais vitesse max
        mental=1.0,
        niveau_entrainement=0.7, # Très bon technique pour compenser la force
        intelligence=149
    )

    ids_joueurs = [integrer_joueur(population, **profil) for profil in (PIERRE, ALICE)]

    print("\nLancement du tournoi Hunger Games...")

    # On passe la liste des IDs à surveiller. `niveaux` garde un échantillon des
    # combattants de chaque round pour la cellule Monte-Carlo.
    niveaux = []
    vainqueur = hunger_games(population, tracked_ids=ids_joueurs, nb_rounds=NB_ROUNDS, niveaux=niveaux)

    print(f"\nDétails du champion : {vainqueur}")


In [ ]:
if __name__ == "__main__":
    # BUGFIX (un seul tirage) : le tournoi ci-dessus n'est qu'UN tirage. Ici, chaque
    # joueur refait son parcours 1000 fois contre les combattants de chaque round.
    print("--- Monte-Carlo : 1000 tournois par joueur ---")
    resultats = [estimer_survie(niveaux, nb_tournois=1000, seed=SEED, **profil) for profil in (PIERRE, ALICE)]
    # graph_survie(*resultats)
